# Core photos with event layers - true scale, full resolution

* Your core photos are used **exactly as you provide them**: not cropped, not shrunk.
* The **depth scale is read automatically from the ruler** in each photo (the white/grey
  10 cm blocks, checked against the 1 cm ticks). The code also finds which end of the photo
  is the core top, so photos taken either way round are handled.
* Events are drawn at their `Depth (in core)` - the depths read on these rulers - in the
  **same colours as the age-depth figure** (E1, E2, …). Cores without an event are left blank.
* The **PDF and SVG** contain every photo at its **original pixel size** with vector event
  drawings - use these in your design canvas. A PNG is also saved.

Run the cells from top to bottom (**Runtime → Run all**), then look at the scale check (Step 5).

In [ ]:
# Step 1: install the Excel reader
!pip install -q openpyxl

In [ ]:
# Step 2: plotting code (no changes needed)
"""
Varve-counted age-depth model figure.

Reads varve counts for one or more cores from an Excel workbook and draws a
figure with linked scales:

  (a) age-depth model: age (yr CE) vs depth (cm), one line per core. Every
      event layer is drawn as a thick bar on its core's curve. Each event that
      is correlated between cores (from an "Events" sheet) is shaded as a
      column hanging from the age axis: as wide as the event's age range across
      the cores, with its lower edge running through each core's event base, so
      it matches both the age and depth scales. Columns are labelled E1, E2, ..., and a table beside the plot gives each event's
      age and depth range across the cores
  (b) thickness through time: one strip per core, on the same age axis as (a),
      with each event's age range shaded in the same colour

Event layers are found from the counts themselves: when the same year is
listed on consecutive rows with increasing depth, the depth between them is
an instantaneous deposit (e.g. a turbidite) inside that year. All other depth
between consecutive years is background varve thickness.

Accepted workbook layouts (detected automatically)
--------------------------------------------------
1. Several cores side by side on one sheet, with the core ID in a title row
   above each block of columns:

       GUAC-22A-1G-1 |                 |                  | | GUAC-23A-1G-1 | ...
       Year          | Depth (in core) | Depth (adjusted) | | Year          | ...
       2023          | 14.80           | 0.00             | | 2023          | ...

   When a core has several depth columns, the one containing --depth-col
   (default "adjusted") is used.

2. One sheet per core (sheet name = core ID) with a year and a depth column.

3. Varve thickness only (no year column): one row per varve from the top, in
   mm. Depth is the cumulative thickness and years count back from --top-year.

Header cells are matched case-insensitively: "year"/"age"/"CE"/"AD",
"depth", "thick". If a core appears on more than one sheet (e.g. an extra
"Events" sheet), only the first is used for the counts. Depths are in cm.

Correlated events (optional): a sheet whose name contains "event", in the same
layout, with two rows per event (its top and base in every core). See
read_events().

Usage
-----
    python plot_age_depth.py Lachua_Varve_counts.xlsx -o age_depth_model.pdf
    python plot_age_depth.py counts.xlsx --xlim 1700 2050 --ylim 90 0 --event-mm 10
"""

import argparse
import io
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon
from matplotlib.ticker import FuncFormatter

PLAIN = FuncFormatter(lambda v, _: f"{v:g}")  # 0.1, 1, 10 instead of 10^-1

# Colours matched to the Excel chart (Office theme, in series order).
CORE_COLORS = [
    "#156082",  # dark teal-blue
    "#E97132",  # orange
    "#196B24",  # dark green
    "#0F9ED5",  # light blue
    "#A02B93",  # purple
    "#4EA72E",  # light green
    "#0E2841",  # navy
    "#8B3A0F",  # brown
    "#5E5E5E",  # grey (extra cores)
    "#C9A100",  # mustard
]

# Soft background shades for correlated events (cycled; neighbours always differ).
EVENT_COLORS = ["#F4A261", "#2A9D8F", "#E9C46A", "#6D8EAD", "#E76F51", "#8AB17D",
                "#B5838D", "#9C89B8"]

YEAR_RE = re.compile(r"year|\bage\b|\bce\b|\bad\b", re.I)
DEPTH_RE = re.compile(r"depth", re.I)
THICK_RE = re.compile(r"thick", re.I)


# ----------------------------------------------------------------------------
# Reading
# ----------------------------------------------------------------------------

def _header_row(raw, max_rows=10):
    """Index of the first row holding both a year header and a depth/thickness header."""
    for i in range(min(max_rows, len(raw))):
        cells = [str(v) for v in raw.iloc[i] if pd.notna(v)]
        if any(YEAR_RE.search(c) for c in cells) and \
                any(DEPTH_RE.search(c) or THICK_RE.search(c) for c in cells):
            return i
    return None


def _column_groups(raw, hdr):
    """Split columns into per-core blocks: [(core_name or None, [col, ...]), ...]."""
    names = raw.iloc[hdr - 1].ffill() if hdr > 0 else pd.Series([None] * raw.shape[1])
    groups, current = [], None
    for c in range(raw.shape[1]):
        label = raw.iat[hdr, c]
        if pd.isna(label):
            current = None  # a blank header cell ends a block
            continue
        name = names.iat[c] if pd.notna(names.iat[c]) else None
        if YEAR_RE.search(str(label)) or current is None or current[0] != name:
            current = (name, [])
            groups.append(current)
        current[1].append(c)
    return groups


def _series(year, depth):
    """Numeric (year, depth) rows in file order; rows with blanks dropped."""
    df = pd.DataFrame({"year": pd.to_numeric(pd.Series(year), errors="coerce").to_numpy(),
                       "depth": pd.to_numeric(pd.Series(depth), errors="coerce").to_numpy()})
    return df.dropna().reset_index(drop=True)


def read_cores(source, top_year=None, depth_col="adjusted", sheet=None):
    """Return {core_id: DataFrame(year, depth)} from an Excel workbook.

    `source` is a file path, bytes, or a file-like object.
    """
    if isinstance(source, (bytes, bytearray)):
        source = io.BytesIO(source)
    book = pd.ExcelFile(source, engine="openpyxl")
    sheets = [sheet] if sheet else book.sheet_names
    cores = {}
    for sheet_name in sheets:
        raw = book.parse(sheet_name, header=None)
        raw = raw.dropna(how="all").dropna(axis=1, how="all").reset_index(drop=True)
        raw.columns = range(raw.shape[1])
        hdr = _header_row(raw) if not raw.empty else None
        if hdr is None:
            print(f"Skipping sheet '{sheet_name}': no Year/Depth header row found.")
            continue
        body = raw.iloc[hdr + 1:]
        groups = [g for g in _column_groups(raw, hdr)
                  if any(YEAR_RE.search(str(raw.iat[hdr, c])) or
                         THICK_RE.search(str(raw.iat[hdr, c])) for c in g[1])]
        for k, (name, cols) in enumerate(groups):
            name = str(name).strip() if name else (sheet_name if len(groups) == 1
                                                   else f"{sheet_name} {k + 1}")
            if name in cores:
                continue  # first occurrence wins (e.g. skip a repeated "Events" sheet)
            labels = {c: str(raw.iat[hdr, c]) for c in cols}
            year_c = [c for c in cols if YEAR_RE.search(labels[c])]
            depth_c = [c for c in cols if DEPTH_RE.search(labels[c])]
            thick_c = [c for c in cols if THICK_RE.search(labels[c])]
            if year_c and depth_c:
                preferred = [c for c in depth_c if depth_col and depth_col.lower() in labels[c].lower()]
                d = (preferred or depth_c)[0]
                df = _series(body[year_c[0]], body[d])
            elif thick_c:
                if top_year is None:
                    raise ValueError(f"'{name}' has only varve thickness; "
                                     "pass top_year (year of the topmost varve).")
                t_cm = pd.to_numeric(body[thick_c[0]], errors="coerce").dropna().to_numpy() / 10
                depth = np.concatenate([[0.0], np.cumsum(t_cm)])
                df = _series(top_year - np.arange(len(depth)), depth)
            else:
                continue
            if len(df) >= 2:
                cores[name] = df
    if not cores:
        raise ValueError("No core data found. Check that each core has a 'Year' and a "
                         "'Depth' column header.")
    return cores


def split_varves_events(df):
    """Split one core's (year, depth) rows into background varves and event layers.

    Returns two DataFrames with columns year, top, base, thickness (mm).
    Consecutive rows with the same year and increasing depth are an event layer;
    the depth between consecutive different years is that varve's thickness.
    """
    year, depth = df["year"].to_numpy(), df["depth"].to_numpy()
    rows_v, rows_e = [], []
    for i in range(1, len(df)):
        inc = depth[i] - depth[i - 1]
        if inc <= 0:
            continue
        rec = (year[i - 1], depth[i - 1], depth[i], inc * 10.0)
        (rows_e if year[i] == year[i - 1] else rows_v).append(rec)
    cols = ["year", "top", "base", "thickness"]
    return pd.DataFrame(rows_v, columns=cols), pd.DataFrame(rows_e, columns=cols)


def read_events(source, depth_col="adjusted", sheet=None):
    """Read correlated event layers from an "Events" sheet, if the workbook has one.

    The sheet uses the same layout as the counts (core ID row, then Year/Depth
    headers). Rows come in pairs, one pair per event: the top row and the base
    row of that event in every core, e.g. rows 1-2 = E1, rows 3-4 = E2. A core
    with no layer for an event has blank cells. Reading stops at the first
    fully blank row.

    Returns [{"name": "E1", "cores": {core_id: (year, top, base)}}, ...], or []
    when there is no such sheet.
    """
    if isinstance(source, (bytes, bytearray)):
        source = io.BytesIO(source)
    if hasattr(source, "seek"):
        source.seek(0)
    book = pd.ExcelFile(source, engine="openpyxl")
    names = [sheet] if sheet else [n for n in book.sheet_names if "event" in n.lower()]
    for sheet_name in names:
        raw = book.parse(sheet_name, header=None)
        raw = raw.dropna(axis=1, how="all")
        raw.columns = range(raw.shape[1])
        hdr = _header_row(raw)
        if hdr is None:
            continue
        series = {}
        for name, cols in _column_groups(raw, hdr):
            labels = {c: str(raw.iat[hdr, c]) for c in cols}
            year_c = [c for c in cols if YEAR_RE.search(labels[c])]
            depth_c = [c for c in cols if DEPTH_RE.search(labels[c])]
            if name is None or not (year_c and depth_c):
                continue
            preferred = [c for c in depth_c if depth_col and depth_col.lower() in labels[c].lower()]
            series[str(name).strip()] = (
                pd.to_numeric(raw[year_c[0]].iloc[hdr + 1:], errors="coerce").to_numpy(),
                pd.to_numeric(raw[(preferred or depth_c)[0]].iloc[hdr + 1:], errors="coerce").to_numpy())
        if not series:
            continue
        years = np.array([y for y, _ in series.values()])
        blank = np.isnan(years).all(axis=0)
        n_rows = int(np.argmax(blank)) if blank.any() else years.shape[1]
        events = []
        for k in range(n_rows // 2):
            per_core = {}
            for core, (y, d) in series.items():
                y1, y2, d1, d2 = y[2 * k], y[2 * k + 1], d[2 * k], d[2 * k + 1]
                if not np.isnan([y1, d1, d2]).any() and d2 > d1:
                    per_core[core] = (y1, d1, d2)
            if per_core:
                events.append({"name": f"E{k + 1}", "cores": per_core})
        if events:
            print(f"Read {len(events)} correlated events from sheet '{sheet_name}'.")
            return events
    return []


def event_envelope(event, top=None):
    """Polygon (list of (age, depth)) enclosing one correlated event in every core.

    Each core's layer is the rectangle [year - 0.5, year + 0.5] x [top, base]
    (a layer fills its whole varve year); the envelope is the convex hull of
    those corners, so it touches every core's event top and base exactly.

    With `top` (a depth, e.g. 0), the polygon is extended straight up to that
    depth: a column as wide as the event's age range across the cores, hanging
    from the age axis, whose lower edge runs through the cores' event bases.
    """
    pts = {(y + dx, d) for y, t, b in event["cores"].values()
           for dx in (-0.5, 0.5) for d in (t, b)}
    if top is not None:
        xs = [x for x, _ in pts]
        pts |= {(min(xs), top), (max(xs), top)}
    pts = sorted(pts)

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower, upper = [], []
    for p in pts:  # Andrew's monotone chain
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)
    return lower[:-1] + upper[:-1]


# ----------------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------------

def _nice_limits(lo, hi, step):
    return np.floor(lo / step) * step, np.ceil(hi / step) * step


def _auto_limits(cores, xlim, ylim, xstep, ystep):
    years = pd.concat([d["year"] for d in cores.values()])
    depths = pd.concat([d["depth"] for d in cores.values()])
    if xlim is None:
        xlim = _nice_limits(years.min(), years.max(), xstep)
    if ylim is None:
        ylim = (_nice_limits(0, depths.max(), ystep)[1], 0)
    return xlim, ylim


def _style_age_depth(ax, xlim, ylim, xstep, ystep):
    ax.set_xlim(xlim)
    ax.set_ylim(max(ylim), min(ylim))  # depth increases downward
    ax.set_xticks(np.arange(min(xlim), max(xlim) + xstep, xstep))
    ax.set_yticks(np.arange(min(ylim), max(ylim) + ystep, ystep))
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.set_xlabel("Age (yr CE)", fontsize=12, labelpad=8)
    ax.set_ylabel("Depth (cm)", fontsize=12)
    ax.grid(color="0.9", lw=0.6)
    ax.set_axisbelow(True)


def plot_age_depth(cores, out_path, xlim=None, ylim=None, style="line",
                   title=None, xstep=50, ystep=10):
    """Single panel: age-depth curves only (like the original Excel chart)."""
    xlim, ylim = _auto_limits(cores, xlim, ylim, xstep, ystep)
    fig, ax = plt.subplots(figsize=(11, 8))
    for i, (name, df) in enumerate(cores.items()):
        ax.plot(df["year"], df["depth"], color=CORE_COLORS[i % len(CORE_COLORS)], lw=1.6,
                label=name, drawstyle="steps-post" if style == "steps" else "default")
    _style_age_depth(ax, xlim, ylim, xstep, ystep)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.6), frameon=False, fontsize=11)
    if title:
        ax.set_title(title, fontsize=13, pad=40)
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved {out_path}")
    return fig


def _event_table(ax, events, cores, depth_label="Depth\n(cm)"):
    """Table of each correlated event's age and depth range across the cores."""
    rows = []
    for ev in events:
        y = [v[0] for v in ev["cores"].values()]
        tops = [v[1] for v in ev["cores"].values()]
        bases = [v[2] for v in ev["cores"].values()]
        age = f"{min(y):.0f}" if min(y) == max(y) else f"{min(y):.0f}–{max(y):.0f}"
        rows.append([ev["name"], age, f"{min(tops):.1f}–{max(bases):.1f}",
                     f"{len(ev['cores'])}/{len(cores)}"])
    ax.axis("off")
    table = ax.table(cellText=rows, colLabels=["Event", "Age\n(yr CE)", depth_label, "Cores"],
                     loc="upper center", cellLoc="center", colWidths=[0.2, 0.32, 0.32, 0.18])
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 0.85)
    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor("0.8")
        cell.set_linewidth(0.5)
        if r == 0:
            cell.set_text_props(fontweight="bold")
            cell.set_facecolor("0.93")
            cell.set_height(cell.get_height() * 1.8)
        elif c == 0:  # event name in that event's shade, as in the plot
            cell.set_facecolor(EVENT_COLORS[(r - 1) % len(EVENT_COLORS)])
            cell.set_alpha(0.45)
            cell.set_text_props(fontweight="bold")


def plot_age_depth_panel(cores, out_path, events=None, xlim=None, ylim=None, style="line",
                         title=None, xstep=50, ystep=10, event_mm=None):
    """(a) age-depth curves with event layers and shaded event columns hanging
    from the age axis, plus a table of event ages and depths; (b) thickness through time,
    one strip per core, with each event's age range shaded."""
    names = list(cores)
    events = events or []
    colors = {n: CORE_COLORS[i % len(CORE_COLORS)] for i, n in enumerate(names)}
    parts = {n: split_varves_events(df) for n, df in cores.items()}
    xlim, ylim = _auto_limits(cores, xlim, ylim, xstep, ystep)
    all_t = pd.concat([p["thickness"] for v_e in parts.values() for p in v_e])
    tlim = (10 ** np.floor(np.log10(all_t.min())), 10 ** np.ceil(np.log10(all_t.max())))

    n = len(names)
    fig = plt.figure(figsize=(12, 8.5 + 0.45 * n))
    outer = fig.add_gridspec(2, 2, width_ratios=[3.2, 1] if events else [1, 0.001],
                             height_ratios=[3, 0.16 * n + 0.4], wspace=0.04, hspace=0.08)
    ax = fig.add_subplot(outer[0, 0])
    strips = outer[1, 0].subgridspec(n, 1, hspace=0)
    axs = [fig.add_subplot(strips[i], sharex=ax) for i in range(n)]

    # (a) age-depth model
    for name, df in cores.items():
        ax.plot(df["year"], df["depth"], color=colors[name], lw=1.5, label=name, zorder=3,
                drawstyle="steps-post" if style == "steps" else "default")
        # every event layer in the counts, drawn as a thick bar over the curve
        evt = parts[name][1]
        ax.vlines(evt["year"], evt["top"], evt["base"], color=colors[name], lw=4.5,
                  alpha=0.45, zorder=2)
    _style_age_depth(ax, xlim, ylim, xstep, ystep)

    # correlated events: a shaded column hanging from the age axis for each event.
    # Its width is the event's age range across the cores (each layer spanning
    # its varve year, +/- 0.5 yr) and its lower edge runs through the cores'
    # event bases, so it matches both the age and the depth scale.
    placed = []
    depth_top = min(ylim)
    for k, ev in enumerate(events):
        color = EVENT_COLORS[k % len(EVENT_COLORS)]
        hull = event_envelope(ev, top=depth_top)
        ax.add_patch(Polygon(hull, closed=True, facecolor=color, alpha=0.2,
                             edgecolor="none", zorder=1))
        ax.add_patch(Polygon(hull, closed=True, facecolor="none", edgecolor=color,
                             alpha=0.9, lw=0.9, zorder=1))
        # coloured tick on the age axis marking the event's age range
        x_lo, x_hi = min(x for x, _ in hull), max(x for x, _ in hull)
        ax.plot([x_lo, x_hi], [depth_top, depth_top], color=color, lw=5,
                solid_capstyle="butt", clip_on=False, zorder=6)
        # label just right of the column's deepest corner, nudged if crowded
        x0, y0 = max(hull, key=lambda p: (p[1], p[0]))
        lx, ly = x0 + 2, y0
        while any(abs(lx - px) < 13 and abs(ly - py) < 1.6 for px, py in placed):
            ly += 1.6
        placed.append((lx, ly))
        if ly != y0:  # leader line from a nudged label back to its column
            ax.plot([x0, lx], [y0, ly], color="0.4", lw=0.5, zorder=4)
        ax.text(lx, ly, ev["name"], fontsize=8, fontweight="bold", color="0.15",
                va="center", zorder=5, clip_on=True,
                bbox=dict(facecolor=color, edgecolor="none", alpha=0.45, pad=0.6))

    tax = None
    if events:
        tax = fig.add_subplot(outer[0, 1])
        _event_table(tax, events, cores)

    # (b) thickness through time, one strip per core
    for a, name in zip(axs, names):
        var, evt = parts[name]
        for k, ev in enumerate(events):  # event age range, same shade as panel a
            yrs = [v[0] for v in ev["cores"].values()]
            a.axvspan(min(yrs) - 0.5, max(yrs) + 0.5, color=EVENT_COLORS[k % len(EVENT_COLORS)],
                      alpha=0.28, lw=0, zorder=0)
        a.fill_between(var["year"], tlim[0], var["thickness"], step="mid",
                       color=colors[name], alpha=0.25, lw=0)
        a.plot(var["year"], var["thickness"], color=colors[name], lw=0.7, drawstyle="steps-mid")
        if len(evt):
            a.vlines(evt["year"], tlim[0], evt["thickness"], color=colors[name], lw=1.2)
            a.plot(evt["year"], evt["thickness"], "o", ms=3, color=colors[name])
        a.set_yscale("log")
        a.set_ylim(tlim)
        a.set_yticks([v for v in (1, 10, 100) if tlim[0] < v < tlim[1]])
        a.yaxis.set_major_formatter(PLAIN)
        a.minorticks_off()
        a.tick_params(axis="y", labelsize=7, length=2, pad=1)
        a.text(1.005, 0.5, name, transform=a.transAxes, va="center", fontsize=9,
               color=colors[name])
        a.grid(axis="x", color="0.9", lw=0.6)
        a.set_axisbelow(True)
        if event_mm:
            a.axhline(event_mm, color="0.35", ls="--", lw=0.7)
        if a is not axs[-1]:
            a.tick_params(labelbottom=False, bottom=False)
    axs[-1].set_xlabel("Age (yr CE)", fontsize=12)
    axs[n // 2].set_ylabel("Thickness\n(mm, log)", fontsize=10,
                           rotation=0, ha="right", va="center", labelpad=10)

    for a, label in ((ax, "a"), (axs[0], "b")):
        a.text(0.012, 0.985 if a is ax else 0.92, label, transform=a.transAxes,
               fontsize=13, fontweight="bold", va="top", zorder=6,
               bbox=dict(facecolor="white", edgecolor="none", pad=1.5))

    handles = ax.get_legend_handles_labels()[0] + [
        Line2D([], [], color="0.5", lw=4.5, alpha=0.45, label="Event layer")]
    if events:
        handles.append(Patch(facecolor=EVENT_COLORS[0], edgecolor=EVENT_COLORS[0], alpha=0.4,
                             label="Event age range\n(from age axis to\nevent base)"))
    if tax is not None:  # below the event table, clear of the curves and labels
        tax.legend(handles=handles, loc="lower center", frameon=True, edgecolor="0.8",
                   fontsize=8, ncol=1, title="Core", title_fontsize=8.5)
    else:
        ax.legend(handles=handles, loc="lower right", frameon=True, framealpha=0.9,
                  edgecolor="0.8", fontsize=9, ncol=2, title="Core", title_fontsize=9)
    if title:
        fig.suptitle(title, fontsize=14)

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved {out_path}")
    return fig


def summarize(cores):
    for name, df in cores.items():
        var, evt = split_varves_events(df)
        print(f"  {name}: {df['year'].min():.0f}-{df['year'].max():.0f} CE, "
              f"{df['depth'].max():.1f} cm, {len(var)} varves "
              f"(median {var['thickness'].median():.1f} mm), {len(evt)} event layers "
              f"(max {evt['thickness'].max() if len(evt) else 0:.0f} mm)")


"""
Core photos with event layers, at the photos' own true scale.

Each photo is used exactly as provided (not cropped): the whole image,
including any ruler and background, is placed on one shared depth axis (cm)
using a linear depth scale for that photo,

    depth (cm) = depth at the photo's top edge + pixel position / pixels per cm

where the pixel position is counted along the core from the photo's core-top
edge. The same pixels-per-cm sets the photo's width, and the plot uses equal x
and y scales, so the photo keeps its real proportions (1 cm across = 1 cm
down). Event layers are then drawn at their depths on that same scale, with a
coloured bar beside the core, and the same event is linked between
neighbouring cores. Event colours are those of plot_age_depth.py, so E1, E2,
... match the age-depth figure.

Use plot_scale_check() first: it draws the code's depth ticks over each full
photo so you can see that they sit on the photo's own ruler.

Depth scale of each photo (first available wins)
------------------------------------------------
1. Two ruler readings: two depths read off the ruler in the photo
   (Ref1 cm, Ref2 cm) and the pixel positions where they are (Ref1 px,
   Ref2 px). Pixel positions are the coordinates your image viewer shows
   along the core: the y coordinate for an upright photo, the x coordinate for
   a sideways one.
2. One ruler reading plus Px per cm (or the photo's stored DPI).
3. Px per cm (or stored DPI) plus Top (cm), the depth at the photo's top edge
   (default 0).
4. Top (cm) and Bottom (cm): the depths at the photo's top and bottom edges.
There is deliberately no fallback: a photo without one of these is refused,
because guessing its scale (e.g. stretching it to the counted depths) puts the
sediment, and so every event, at the wrong depth.

Inputs
------
- Core photos (JPG, PNG, TIFF) as provided. A photo is matched to its core
  when the file name contains the core ID ("GUAC-22A-1G-1.jpg",
  "guac_22a_1g_1.tif", ...). Sideways photos are turned upright; set Top side
  (top/bottom/left/right) for the edge where the core top is (default: top for
  upright photos, left for sideways ones).
- Photo depth table (optional, CSV/Excel): Core and any of Ref1 px, Ref1 cm,
  Ref2 px, Ref2 cm, Px per cm, Top (cm), Bottom (cm), Top side, File.
- Event depths: the "Events" sheet of the varve workbook (two rows per event:
  top then base; the "Depth (in core)" column by default), or a table with
  Core, Event, Top (cm), Base (cm). Use the same depth reference as the photo
  rulers (e.g. both measured from the top of the core liner).

Usage
-----
    python plot_core_photos.py --photos photos/*.jpg --photo-depths photo_depths.csv \
        --events Lachua_Varve_counts.xlsx -o core_photos_events.pdf \
        --check scale_check.png
"""

import argparse
import io
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Polygon, Rectangle
from PIL import Image


Image.MAX_IMAGE_PIXELS = None  # core scans can be very large
CHECK_PIXELS_LONG_SIDE = 6000  # the scale-check figure uses a smaller copy (faster)
SPEC_KEYS = ("ref1_px", "ref1_cm", "ref2_px", "ref2_cm", "px_per_cm", "top", "bottom",
             "top_side", "file")


def _key(text):
    """Core ID / file name reduced to lowercase letters and digits for matching."""
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


def event_color(name, k):
    """Same colour as plot_age_depth.py: E1 -> first colour, E2 -> second, ..."""
    m = re.search(r"(\d+)", str(name))
    i = int(m.group(1)) - 1 if m else k
    return EVENT_COLORS[i % len(EVENT_COLORS)]


# ----------------------------------------------------------------------------
# Reading
# ----------------------------------------------------------------------------

def _read_table(source):
    if isinstance(source, pd.DataFrame):
        return source
    if str(source).lower().endswith((".csv", ".txt")):
        return pd.read_csv(source)
    return pd.read_excel(source, engine="openpyxl")


def _col(columns, test):
    return next((c for c in columns if test(re.sub(r"[^a-z0-9]", "", str(c).lower()))), None)


def read_photo_depths(source):
    """{core: spec} from a CSV/Excel table or DataFrame (see module docstring)."""
    df = _read_table(source)
    cols = {
        "core": _col(df.columns, lambda s: s.startswith("core")),
        "ref1_px": _col(df.columns, lambda s: s.startswith("ref1") and "px" in s),
        "ref1_cm": _col(df.columns, lambda s: s.startswith("ref1") and "cm" in s),
        "ref2_px": _col(df.columns, lambda s: s.startswith("ref2") and "px" in s),
        "ref2_cm": _col(df.columns, lambda s: s.startswith("ref2") and "cm" in s),
        "px_per_cm": _col(df.columns, lambda s: "percm" in s or s.startswith("pxper")),
        "top": _col(df.columns, lambda s: s.startswith("top") and "side" not in s),
        "bottom": _col(df.columns, lambda s: s.startswith("bottom") or s.startswith("base")),
        "top_side": _col(df.columns, lambda s: "side" in s),
        "file": _col(df.columns, lambda s: s.startswith("file") or s.startswith("photo")),
    }
    if cols["core"] is None:
        raise ValueError("The photo depth table needs a 'Core' column.")
    out = {}
    for _, r in df.iterrows():
        if pd.isna(r[cols["core"]]):
            continue
        spec = {}
        for k in SPEC_KEYS:
            c = cols[k]
            v = r[c] if c is not None else None
            if v is None or (not isinstance(v, str) and pd.isna(v)) or str(v).strip() == "":
                spec[k] = None
            elif k in ("top_side", "file"):
                spec[k] = str(v).strip().lower() if k == "top_side" else str(v).strip()
            else:
                spec[k] = float(v)
        out[str(r[cols["core"]]).strip()] = spec
    return out


def read_event_depths(source, depth_col="in core"):
    """Events as [{"name": "E1", "cores": {core: (year or None, top, base)}}, ...].

    Accepts the Events-sheet layout read by plot_age_depth.read_events, or a
    long table with columns Core, Event, Top (cm), Base (cm).
    """
    if not isinstance(source, pd.DataFrame):
        try:
            events = read_events(source, depth_col=depth_col)
        except Exception:
            events = []
        if events:
            return events
    df = _read_table(source)
    core_c = _col(df.columns, lambda s: s.startswith("core"))
    ev_c = _col(df.columns, lambda s: s.startswith("event"))
    top_c = _col(df.columns, lambda s: s.startswith("top"))
    base_c = _col(df.columns, lambda s: s.startswith("base") or s.startswith("bottom"))
    if None in (core_c, ev_c, top_c, base_c):
        raise ValueError("Event file needs an 'Events' sheet (Year/Depth layout) or columns "
                         "Core, Event, Top (cm), Base (cm).")
    events = {}
    for _, r in df.dropna(subset=[core_c, ev_c, top_c, base_c]).iterrows():
        name = str(r[ev_c]).strip()
        name = f"E{int(float(name))}" if re.fullmatch(r"\d+(\.0)?", name) else name
        events.setdefault(name, {})[str(r[core_c]).strip()] = (None, float(r[top_c]), float(r[base_c]))
    return [{"name": n, "cores": c} for n, c in events.items()]


def match_photos(files, cores):
    """{core: file} by finding each core ID inside the file names."""
    out = {}
    for core in cores:
        hits = [f for f in files if _key(core) in _key(Path(f).stem)]
        if hits:
            out[core] = sorted(hits, key=lambda f: len(Path(f).stem))[0]
    return out


# ----------------------------------------------------------------------------
# Photo geometry and depth scale
# ----------------------------------------------------------------------------

def photo_geometry(path, top_side=None):
    """Size of the photo as provided, and which edge is the core top.

    Returns (width_px, height_px, side, length_px, across_px), where length is
    measured along the core and across perpendicular to it.
    """
    w, h = Image.open(path).size
    side = (top_side or ("left" if w > h else "top")).lower()
    if side not in ("top", "bottom", "left", "right"):
        raise ValueError(f"Top side must be top, bottom, left or right (got {top_side!r}).")
    length, across = (h, w) if side in ("top", "bottom") else (w, h)
    return w, h, side, length, across


def along_core(px, side, w, h):
    """Viewer coordinate (y for upright, x for sideways) -> pixels from the core-top edge."""
    return {"top": px, "bottom": h - px, "left": px, "right": w - px}[side]


def photo_px_per_cm(path):
    """Pixels per cm from the photo's stored resolution (DPI), or None.

    72 or 96 DPI is a screen default that says nothing about the real scale,
    so values of 100 DPI or less are ignored.
    """
    try:
        dpi = Image.open(path).info.get("dpi")
    except Exception:
        return None
    if not dpi:
        return None
    d = float(dpi[1] if len(dpi) > 1 else dpi[0])
    return d / 2.54 if d > 100 else None


def _oriented_gray(path, side):
    """Grey photo turned so the core top is on the LEFT (core axis along x)."""
    img = Image.open(path).convert("L")
    rotate = {"left": 0, "top": 90, "right": 180, "bottom": -90}[side]
    if rotate:
        img = img.rotate(rotate, expand=True)
    return np.asarray(img, float)


def _ruler_band(a, bottom=True):
    """Rows of the ruler strip: beyond the darkest row near that photo edge."""
    h = a.shape[0]
    if not bottom:
        a = a[::-1]
    band = a[int(h * 0.7):]
    dark = int(np.argmin(np.median(band, axis=1)))
    return band[dark + 2:]


def _block_edges(rows):
    """Sub-pixel x positions of white/grey block changes along the ruler."""
    p = np.median(rows, axis=0)
    k = max(3, len(p) // 400)
    ps = np.convolve(p, np.ones(k) / k, mode="same")
    lo, hi = np.percentile(ps, 10), np.percentile(ps, 90)
    thr = (lo + hi) / 2
    idx = np.where(np.diff((ps > thr).astype(int)) != 0)[0]
    idx = idx[(idx > k) & (idx < len(ps) - k - 1)]
    edges = idx + (thr - ps[idx]) / (ps[idx + 1] - ps[idx])
    rising = ps[idx + 1] > ps[idx]
    return edges, rising, hi - lo


def _tick_px_per_cm(rows, guess):
    """Independent px/cm from the repeating 1 cm tick pattern (Fourier peak)."""
    p = rows.mean(axis=0)
    win = max(3, int(guess * 2))
    p = p - np.convolve(p, np.ones(win) / win, mode="same")
    n = len(p) * 16
    F = np.abs(np.fft.rfft(p * np.hanning(len(p)), n=n))
    f = np.fft.rfftfreq(n)
    band = (f > 1 / (guess * 1.3)) & (f < 1 / (guess * 0.7))
    return 1 / f[band][np.argmax(F[band])] if band.any() else None


def _fit_ruler(a, block_cm=10.0):
    """Fit the ruler in an image oriented with the core top on the LEFT.

    Returns the best fit over both long sides of the photo, as a dict with
    x0 (pixel of ruler 0), ppc (px per cm), the edges used and a direction
    score: on these rulers 0-10 cm is grey, 10-20 white, 20-30 grey, ..., so
    going down-core grey->white changes fall at 10, 30, 50 ... cm and
    white->grey at 20, 40 ... cm. The score is the fraction of edges that
    follow this pattern (1 = ruler reads the right way, ~0 = reversed).
    """
    best = None
    for bottom in (True, False):  # ruler on either long side of the core
        rows = _ruler_band(a, bottom)
        if rows.shape[0] < 3:
            continue
        edges, rising, contrast = _block_edges(rows)
        if len(edges) < 4:
            continue
        gap = np.median(np.diff(edges))
        keep = np.array([any(abs(abs(edges[j] - edges[i]) - gap) < 0.12 * gap
                             for j in (i - 1, i + 1) if 0 <= j < len(edges))
                         for i in range(len(edges))])
        e, rs = edges[keep], rising[keep]
        if len(e) < 4:
            continue
        n = np.round(e / gap)
        for _ in range(5):  # fit, drop the worst misfit > 1.5 px, refit
            A = np.column_stack([np.ones(len(e)), n * block_cm, np.where(rs, 1.0, -1.0)])
            coef, *_ = np.linalg.lstsq(A, e, rcond=None)
            resid = e - A @ coef
            worst = int(np.argmax(np.abs(resid)))
            if abs(resid[worst]) <= 1.5 or len(e) <= 4:
                break
            e, rs, n = np.delete(e, worst), np.delete(rs, worst), np.delete(n, worst)
        direction = float(np.mean(np.where(rs, n % 2 == 1, n % 2 == 0)))
        score = contrast * len(e)
        if best is None or score > best["score"]:
            best = {"score": score, "x0": coef[0], "ppc": coef[1], "edges": e, "n": n,
                    "resid": resid, "rows": rows, "direction": direction}
    return best


def detect_ruler_scale(path, top_side=None, block_cm=10.0, ruler_start_cm=0.0):
    """Read the depth scale - and which end is the core top - from the ruler.

    The ruler alternates grey and white blocks every `block_cm` (10 cm). The
    block edges along the ruler strip (beside the core) are fitted with a
    straight line (depth -> pixel) after removing edges that do not fit (e.g. a
    differently printed 100 cm mark), and the scale is checked against the 1 cm
    tick spacing measured independently. If `top_side` is not given, both ends
    of the photo are tried and the one where the ruler reads the right way
    (grey 0-10 cm, white 10-20 cm, ...) is taken as the core top. The ruler is
    assumed to start at `ruler_start_cm` at the core-top edge of the photo.

    Returns (depth at the photo's core-top edge, px per cm, report dict with
    "side" = the core-top edge), or (None, None, report) if no ruler is found.
    """
    w, h = Image.open(path).size
    sides = [top_side.lower()] if top_side else (["left", "right"] if w > h else ["top", "bottom"])
    fits = {}
    for side in sides:
        fit = _fit_ruler(_oriented_gray(path, side), block_cm)
        if fit:
            fits[side] = fit
    if not fits:
        return None, None, {"ok": False, "why": "no ruler found"}
    # core top = the end where the ruler reads the right way and its 0 mark sits at
    # the photo edge (on these photos the ruler starts exactly at the core-top edge)
    edge_cm = {k: abs(v["x0"]) / v["ppc"] for k, v in fits.items()}
    side = max(fits, key=lambda k: (round(fits[k]["direction"], 1), -edge_cm[k]))
    f = fits[side]
    tick = _tick_px_per_cm(f["rows"], f["ppc"])
    report = {
        "ok": True, "side": side, "px_per_cm": f["ppc"], "block_edges_used": len(f["edges"]),
        "direction_score": f["direction"],
        "ruler0_from_edge_cm": edge_cm[side],
        "other_end": None if len(fits) < 2 else {
            "direction_score": next(v["direction"] for k, v in fits.items() if k != side),
            "ruler0_from_edge_cm": next(edge_cm[k] for k in fits if k != side)},
        "marks_cm": f"{ruler_start_cm + f['n'].min() * block_cm:g}-"
                    f"{ruler_start_cm + f['n'].max() * block_cm:g}",
        "max_misfit_mm": float(np.abs(f["resid"]).max() / f["ppc"] * 10),
        "tick_px_per_cm": tick,
        "scale_agreement_pct": None if tick is None else float(abs(tick - f["ppc"]) / f["ppc"] * 100),
        "ruler0_offset_px": float(f["x0"]),
    }
    # x0 = pixel position (from the core-top edge) of ruler mark ruler_start_cm
    return ruler_start_cm - f["x0"] / f["ppc"], f["ppc"], report


def photo_scale(path, spec):
    """Linear depth scale of one photo: (depth at top edge, px per cm, source).

    `spec` holds any of ref1_px/ref1_cm/ref2_px/ref2_cm, px_per_cm, top,
    bottom, top_side (see module docstring for the order used).
    """
    w, h, side, length, _ = photo_geometry(path, spec.get("top_side"))
    g = lambda k: spec.get(k)
    refs = [(along_core(g(f"ref{i}_px"), side, w, h), g(f"ref{i}_cm")) for i in (1, 2)
            if g(f"ref{i}_px") is not None and g(f"ref{i}_cm") is not None]
    ppc_given, ppc_dpi = g("px_per_cm"), photo_px_per_cm(path)
    if len(refs) == 2:
        (p1, d1), (p2, d2) = refs
        if d1 == d2 or p1 == p2:
            raise ValueError(f"{Path(path).name}: the two ruler readings must differ.")
        ppc = (p2 - p1) / (d2 - d1)
        if ppc <= 0:
            raise ValueError(f"{Path(path).name}: ruler readings give depth decreasing "
                             "down the core; check Top side or the pixel values.")
        return d1 - p1 / ppc, ppc, "ruler (2 readings)"
    if len(refs) == 1 and (ppc_given or ppc_dpi):
        (p1, d1), ppc = refs[0], ppc_given or ppc_dpi
        return d1 - p1 / ppc, ppc, "ruler (1 reading) + " + ("px per cm" if ppc_given else "photo DPI")
    top = g("top") if g("top") is not None else 0.0
    if ppc_given:
        return top, ppc_given, "px per cm + top depth"
    if g("bottom") is not None:
        if g("bottom") <= top:
            raise ValueError(f"{Path(path).name}: Bottom (cm) must be deeper than Top (cm).")
        return top, length / (g("bottom") - top), "top + bottom depths"
    if ppc_dpi:
        return top, ppc_dpi, "photo DPI + top depth"
    t_auto, ppc_auto, rep = detect_ruler_scale(path, spec.get("top_side"))
    if ppc_auto:
        spec["top_side"] = rep["side"]  # core-top end found from the ruler
        return t_auto, ppc_auto, f"ruler (read automatically; core top = {rep['side']} edge)"
    return None, None, "NO DEPTH SCALE - add two ruler readings"


def resolve_photo_depths(cores, photos, photo_depths=None):
    """Depth scale for every core's photo.

    Returns ({core: {"file", "top_side", "top", "px_per_cm", "bottom"}},
    summary DataFrame showing where each scale came from).
    """
    photo_depths = photo_depths or {}
    scales, rows, missing = {}, [], []
    for core in cores:
        spec = {k: None for k in SPEC_KEYS}
        spec.update(photo_depths.get(core, {}))
        path = photos.get(core) or spec.get("file")
        if not path or not Path(path).exists():
            rows.append({"Core": core, "Photo": "-", "Px per cm": None, "Top edge (cm)": None,
                         "Bottom edge (cm)": None, "Depth scale from": "no photo (blank column)"})
            continue
        top, ppc, source = photo_scale(path, spec)
        _, _, side, length, _ = photo_geometry(path, spec.get("top_side"))
        if ppc is None:
            missing.append(core)
            rows.append({"Core": core, "Photo": Path(path).name, "Px per cm": None,
                         "Top edge (cm)": None, "Bottom edge (cm)": None,
                         "Depth scale from": source})
            continue
        scales[core] = {"file": path, "top_side": side, "top": top, "px_per_cm": ppc,
                        "bottom": top + length / ppc}
        rows.append({"Core": core, "Photo": Path(path).name, "Px per cm": round(ppc, 3),
                     "Top edge (cm)": round(top, 2), "Bottom edge (cm)": round(top + length / ppc, 2),
                     "Depth scale from": source})
    summary = pd.DataFrame(rows)
    if missing:
        raise ValueError(
            "No depth scale for: " + ", ".join(missing) + ".\nEnter two ruler readings "
            "(depth in cm and pixel position of two marks on the photo's ruler) for each, "
            "or Px per cm with Top (cm), or Top and Bottom (cm).\n" + summary.to_string(index=False))
    return scales, summary


def load_upright(path, side, max_px=None):
    """The whole photo (nothing cropped), turned so the core top is at the top edge.

    Turning is an exact pixel transpose (no resampling). The photo is kept at its
    original size unless `max_px` (long side) asks for a smaller copy.
    """
    img = Image.open(path).convert("RGB")
    turn = {"top": None, "left": Image.Transpose.ROTATE_270, "right": Image.Transpose.ROTATE_90,
            "bottom": Image.Transpose.ROTATE_180}[side]
    if turn is not None:
        img = img.transpose(turn)
    if max_px and max(img.size) > max_px:
        scale = max_px / max(img.size)
        img = img.resize((round(img.size[0] * scale), round(img.size[1] * scale)), Image.LANCZOS)
    return img


def show_photo_pixels(path, start_px=None, end_px=None, out_path=None, height_in=10):
    """Show a photo as provided with its pixel coordinates, to read ruler marks.

    The axes are pixel coordinates exactly as an image viewer reports them
    (x across, y down). start_px/end_px zoom to part of the photo along its
    long side, e.g. around the ruler's 0 and 80 cm marks.
    """
    img = Image.open(path)
    w, h = img.size
    sideways = w > h
    fig, ax = plt.subplots(figsize=(height_in, height_in * 0.35) if sideways
                           else (height_in * 0.35, height_in))
    ax.imshow(img.convert("RGB"), interpolation="nearest")
    lo, hi = start_px or 0, end_px or (w if sideways else h)
    if sideways:
        ax.set_xlim(lo, hi)
    else:
        ax.set_ylim(hi, lo)
    ax.minorticks_on()
    ax.grid(which="major", color="cyan", lw=0.5, alpha=0.7)
    ax.grid(which="minor", color="cyan", lw=0.3, alpha=0.3)
    ax.set_title(f"{Path(path).name}  ({w} x {h} px)  - read ruler marks as "
                 f"{'x' if sideways else 'y'} pixel coordinates", fontsize=9)
    if out_path:
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
    return fig


# ----------------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------------

def _layout(cores, scales, events, gap_cm, default_width_cm):
    """x position and depth extent (cm) of each core's column."""
    columns, x = [], 0.0
    for core in cores:
        s = scales.get(core)
        if s:
            _, _, _, length, across = photo_geometry(s["file"], s["top_side"])
            width = across / s["px_per_cm"]  # same scale across as along the core
            top, bottom = s["top"], s["bottom"]
        else:
            depths = [d for ev in events if core in ev["cores"] for d in ev["cores"][core][1:]]
            width, top, bottom = default_width_cm, 0.0, (max(depths) if depths else 10.0)
        columns.append({"core": core, "x0": x, "x1": x + width, "top": top, "bottom": bottom,
                        "scale": s})
        x += width + gap_cm
    return columns, x - gap_cm


def _draw_photo(ax, col, max_px=None, interpolation="none"):
    s = col["scale"]
    if s:
        # interpolation="none": PDF/SVG embed the photo's own pixels, not a resampled copy
        ax.imshow(load_upright(s["file"], s["top_side"], max_px),
                  extent=(col["x0"], col["x1"], col["bottom"], col["top"]),
                  interpolation=interpolation, zorder=1)
    else:
        ax.add_patch(Rectangle((col["x0"], col["top"]), col["x1"] - col["x0"],
                               col["bottom"] - col["top"], facecolor="0.92", edgecolor="0.6",
                               lw=0.6, ls="--", zorder=1))
    ax.add_patch(Rectangle((col["x0"], col["top"]), col["x1"] - col["x0"],
                           col["bottom"] - col["top"], facecolor="none", edgecolor="0.25",
                           lw=0.8, zorder=4))


def _depth_axis(ax, ylim, ystep):
    ax.set_ylim(max(ylim), min(ylim))
    ax.set_aspect("equal")  # 1 cm across = 1 cm down
    ax.set_yticks(np.arange(min(ylim), max(ylim) + ystep / 2, ystep))
    ax.set_ylabel("Depth (cm)", fontsize=12)
    ax.set_xticks([])
    for side in ("top", "right", "bottom"):
        ax.spines[side].set_visible(False)


def _auto_ylim(columns, ystep):
    lo = min(c["top"] for c in columns)
    hi = max(c["bottom"] for c in columns)
    return np.ceil(hi / ystep) * ystep, np.floor(lo / ystep) * ystep


def plot_core_photos(scales, events, out_path, cores=None, ylim=None, gap_cm=4.0,
                     default_width_cm=7.0, ystep=5, fill_alpha=0.18, link_alpha=0.35,
                     height_in=12, title=None, table=True, png_dpi="native",
                     max_png_megapixels=400):
    """Full photos side by side at true scale with the event layers.

    scales: from resolve_photo_depths(); events: from read_event_depths().

    Resolution: in a PDF or SVG every photo is embedded at its ORIGINAL pixel
    size (nothing resampled or cropped), and the event drawing stays vector -
    best for editing in a design canvas. For a PNG, png_dpi="native" picks the
    dpi that keeps the photos' own pixels per cm, limited to max_png_megapixels
    (all 8 full-size cores side by side can be several hundred megapixels); a
    number sets the dpi directly.
    """
    vector = str(out_path).lower().endswith((".pdf", ".svg", ".eps"))
    cores = cores or list(dict.fromkeys(list(scales) + [c for e in events for c in e["cores"]]))
    columns, total_w = _layout(cores, scales, events, gap_cm, default_width_cm)
    ylim = ylim or _auto_ylim(columns, ystep)
    span = max(ylim) - min(ylim)
    fig_w = max(6.0, height_in * (total_w + 6) / span + 2.5)
    fig, ax = plt.subplots(figsize=(fig_w, height_in))

    for col in columns:
        _draw_photo(ax, col, interpolation="none" if vector else "antialiased")
        ax.text((col["x0"] + col["x1"]) / 2, min(ylim) - 0.012 * span, col["core"],
                ha="left", va="bottom", rotation=40, rotation_mode="anchor", fontsize=9)

    by_core = {c["core"]: c for c in columns}
    bar_w = min(0.9, gap_cm * 0.22)
    placed = []
    for k, ev in enumerate(events):
        color = event_color(ev["name"], k)
        present = [c for c in cores if c in ev["cores"]]
        for core in present:
            col, (_, t, b) = by_core[core], ev["cores"][core]
            ax.add_patch(Rectangle((col["x0"], t), col["x1"] - col["x0"], b - t,
                                   facecolor=color, alpha=fill_alpha, edgecolor="none", zorder=2))
            for d in (t, b):  # event top and base lines across the photo
                ax.plot([col["x0"], col["x1"]], [d, d], color=color, lw=1.4, zorder=3)
            ax.add_patch(Rectangle((col["x1"], t), bar_w, b - t, facecolor=color,
                                   edgecolor="none", zorder=3))
        # band linking the event to the next core that has it; a band that skips
        # cores (event missing there) runs behind them, never over a photo
        for c1, c2 in zip(present, present[1:]):
            a, b_ = by_core[c1], by_core[c2]
            (_, t1, s1), (_, t2, s2) = ev["cores"][c1], ev["cores"][c2]
            adjacent = cores.index(c2) - cores.index(c1) == 1
            ax.add_patch(Polygon([(a["x1"] + bar_w, t1), (b_["x0"], t2), (b_["x0"], s2),
                                  (a["x1"] + bar_w, s1)], closed=True, facecolor=color,
                                 alpha=link_alpha if adjacent else link_alpha * 0.6,
                                 edgecolor=color, lw=0.5, zorder=2 if adjacent else 0.5))
        if present:  # label right of the last core with this event, nudged if crowded
            col, (_, t, b) = by_core[present[-1]], ev["cores"][present[-1]]
            lx, ly = col["x1"] + bar_w + 0.4, (t + b) / 2
            while any(abs(lx - px) < 3 and abs(ly - py) < 0.018 * span for px, py in placed):
                ly += 0.018 * span
            placed.append((lx, ly))
            if ly != (t + b) / 2:
                ax.plot([col["x1"] + bar_w, lx], [(t + b) / 2, ly], color="0.4", lw=0.5, zorder=4)
            ax.text(lx, ly, ev["name"], fontsize=7.5, fontweight="bold", va="center",
                    color="0.15", zorder=5, clip_on=False,
                    bbox=dict(facecolor=color, edgecolor="none", alpha=0.45, pad=0.6))

    ax.set_xlim(-1.5, total_w + 5)
    _depth_axis(ax, ylim, ystep)
    ax.grid(axis="y", color="0.88", lw=0.5)
    ax.set_axisbelow(True)
    if table and events:  # same event table and colours as the age-depth figure
        fig.canvas.draw()
        pos = ax.get_position()
        tax = fig.add_axes([pos.x1 + 0.01, pos.y0 + 0.35 * pos.height, 0.22, 0.65 * pos.height])
        all_cores = list(dict.fromkeys(c for e in events for c in e["cores"]))  # whole workbook
        _event_table(tax, events, all_cores, depth_label="Depth in\ncore (cm)")
    if title:
        ax.set_title(title, fontsize=13, pad=60)
    dpi = 300
    if not vector:
        fig.canvas.draw()
        in_per_cm = ax.get_window_extent().height / fig.dpi / span
        ppc_max = max((c["scale"]["px_per_cm"] for c in columns if c["scale"]), default=0)
        cap = np.sqrt(max_png_megapixels * 1e6 / (fig.get_figwidth() * fig.get_figheight()))
        dpi = (min(ppc_max / in_per_cm, cap) if png_dpi == "native" else float(png_dpi)) if ppc_max else 300
        dpi = max(dpi, 150)
        kept = dpi * in_per_cm
        print(f"PNG at {dpi:.0f} dpi = {kept:.1f} px per cm of core "
              f"(photos have up to {ppc_max:.1f} px/cm"
              + (")" if kept >= ppc_max - 0.5 else "; use the PDF/SVG for full resolution)"))
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    print(f"Saved {out_path}")
    return fig


def plot_scale_check(scales, events, out_path, cores=None, spec=None, height_in=14):
    """Check figure: each full photo with the code's depth ticks drawn over it.

    Red ticks every 1 cm (longer and labelled every 5 cm) are drawn from the
    depth scale the code uses; they should sit on the photo's own ruler marks.
    Green crosses mark the ruler readings you entered, and coloured lines the
    event tops and bases.
    """
    cores = [c for c in (cores or list(scales)) if c in scales]
    spec = spec or {}
    columns, total_w = _layout(cores, scales, events, 3.0, 7.0)
    ylim = _auto_ylim(columns, 5)
    span = max(ylim) - min(ylim)
    fig, ax = plt.subplots(figsize=(max(6.0, height_in * (total_w + 4) / span + 2), height_in))
    for col in columns:
        _draw_photo(ax, col, max_px=CHECK_PIXELS_LONG_SIDE, interpolation="antialiased")
        s, x0, x1 = col["scale"], col["x0"], col["x1"]
        for d in np.arange(np.ceil(col["top"]), np.floor(col["bottom"]) + 1):
            major = d % 5 == 0
            ax.plot([x1 - (1.2 if major else 0.6), x1], [d, d], color="red",
                    lw=0.9 if major else 0.5, zorder=6)
            if major:  # faint line across the whole photo, through the photo's own ruler
                ax.plot([x0, x1], [d, d], color="red", lw=0.4, alpha=0.55, zorder=6)
                ax.text(x1 + 0.15, d, f"{d:g}", color="red", fontsize=6.5, va="center", zorder=6)
        sp = spec.get(col["core"], {}) or {}
        for i in (1, 2):
            if sp.get(f"ref{i}_cm") is not None and sp.get(f"ref{i}_px") is not None:
                ax.plot((x0 + x1) / 2, sp[f"ref{i}_cm"], "+", color="lime", ms=12, mew=1.8, zorder=7)
        for k, ev in enumerate(events):
            if col["core"] in ev["cores"]:
                _, t, b = ev["cores"][col["core"]]
                for d in (t, b):
                    ax.plot([x0, x1], [d, d], color=event_color(ev["name"], k), lw=0.8, zorder=5)
        ax.text((x0 + x1) / 2, min(ylim) - 0.012 * span,
                f"{col['core']}\n{s['px_per_cm']:.2f} px/cm", ha="center", va="bottom", fontsize=8)
    ax.set_xlim(-1, total_w + 2)
    _depth_axis(ax, ylim, 5)
    ax.set_title("Scale check: red ticks = depth scale used by the code; they should sit on "
                 "the photo's ruler", fontsize=10, pad=34)
    fig.savefig(out_path, dpi=250, bbox_inches="tight")
    print(f"Saved {out_path}")
    return fig


In [ ]:
# Step 3: upload the core photos as they are (select all 8 files at once)
# Each file name must contain its core ID, e.g. GUAC-22A-1G-1.jpg or GUAC-22A-1G-1-W.tif
from google.colab import files
import os

os.makedirs("photos", exist_ok=True)
for name, data in files.upload().items():
    with open(os.path.join("photos", name), "wb") as f:
        f.write(data)
PHOTO_FILES = sorted(os.path.join("photos", f) for f in os.listdir("photos"))
for f in PHOTO_FILES:
    print(f"{f}: {Image.open(f).size[0]} x {Image.open(f).size[1]} px")

In [ ]:
# Step 4: upload the varve workbook (with its "Events" sheet), then read the depth scales
uploaded = files.upload()
EVENT_NAME = next(iter(uploaded))
with open(EVENT_NAME, "wb") as f:
    f.write(uploaded[EVENT_NAME])

DEPTH_COLUMN = "in core"   # the depths read on the photo rulers
events = read_event_depths(EVENT_NAME, depth_col=DEPTH_COLUMN)
cores = list(dict.fromkeys(c for e in events for c in e["cores"]))
photos = match_photos(PHOTO_FILES, cores)
print(f"{len(events)} events. Photos found for {len(photos)} of {len(cores)} cores.")

# Manual overrides - normally leave EMPTY: the ruler is read automatically.
#   RULER:    "core ID": (depth 1 cm, pixel 1, depth 2 cm, pixel 2) read off the photo
#   TOP_SIDE: "core ID": "left"/"right"/"top"/"bottom" = edge of the photo with the core top
RULER = {}
TOP_SIDE = {}
spec = {}
for c in cores:
    r = RULER.get(c) or (None,) * 4
    spec[c] = {"ref1_cm": r[0], "ref1_px": r[1], "ref2_cm": r[2], "ref2_px": r[3],
               "top_side": TOP_SIDE.get(c)}
scales, summary = resolve_photo_depths(cores, photos, spec)
for c in cores:   # quality of the automatic ruler reading
    if c in photos and not RULER.get(c):
        _, _, r = detect_ruler_scale(photos[c], TOP_SIDE.get(c))
        print(f"  {c}: core top = {r['side']} edge, {r['px_per_cm']:.2f} px/cm, ruler fit within "
              f"{r['max_misfit_mm']:.2f} mm, 1-cm ticks agree within {r['scale_agreement_pct']:.2f} %")
summary

In [ ]:
# Step 5: SCALE CHECK - red lines (every 5 cm) are the depth scale the code uses, drawn over
# each full photo. They must run through the 5, 10, 15 ... numbers on the photo's own ruler.
plot_scale_check(scales, events, "scale_check.png", cores=cores, spec=spec)
plt.close("all")
from IPython.display import Image as ShowImage, display
display(ShowImage("scale_check.png", width=1000))

In [ ]:
# Step 6: draw the final figure
CORE_ORDER = None    # e.g. ["GUAC-22A-1G-1", "GUAC-23A-1G-1", ...]; None = workbook order
YLIM = None          # e.g. (105, 0) for the depth axis; None = whole photos
FILL_ALPHA = 0.18    # tint of the events on the photos (0 = lines only)
TITLE = None

order = CORE_ORDER or cores
for out in ("core_photos_events.pdf", "core_photos_events.svg", "core_photos_events.png"):
    plot_core_photos(scales, events, out, cores=order, ylim=YLIM, fill_alpha=FILL_ALPHA, title=TITLE)
    plt.close("all")
display(ShowImage("core_photos_events.png", width=1000))

In [ ]:
# Step 7: download - PDF/SVG keep every photo at its original resolution (for your canvas)
files.download("core_photos_events.pdf")
files.download("core_photos_events.svg")
files.download("core_photos_events.png")
files.download("scale_check.png")